# N-Lens System Inversion (Glaser Bell Model)

This notebook fits lens column parameters using the Glaser bell model physics:

1. **Focal length** (Eq. 16): $\displaystyle \frac{1}{f_i} = C_{f,i} \cdot I_{0,i}^2\,(1+w)^2$
2. **Image rotation** (Eq. 17): $\displaystyle \psi_i = K_v \cdot I_{0,i}\,(1+w)$

where wobble $w$ is a relative current perturbation applied to one lens at a time.

## Fit parameters

| Symbol | Meaning | Per-lens? |
|--------|---------|-----------|
| $d_j$ | Propagation distance | yes ($N+1$ total) |
| $I_{0,i}$ | Nominal excitation (ampere-turns) | yes ($N$ total) |
| $C_{f,i}$ | Lens geometry constant (bore, gap, pole-piece) | yes ($N$ total) |
| $K_v$ | Rotation constant $= e\mu_0/(2m_e v)$ | **no — known from voltage** |

## Why no degeneracy

$K_v$ is known $\Rightarrow$ rotation $\psi_i$ directly constrains $I_{0,i}$ $\Rightarrow$ then $C_{f,i}$ is uniquely determined from the ABCD matrix.

In [38]:
import sys
sys.path.insert(0, '../../src')

import jax
import jax.numpy as jnp
import numpy as np
import sympy as sp
from IPython.display import display
from scipy.optimize import least_squares as scipy_least_squares
import time

from temgym_core.transfer_matrices import propagation_matrix, lens_matrix

jax.config.update("jax_enable_x64", True)

print("Imports successful")

Imports successful


## System Parameters

Define the true system and experimental configuration used to generate synthetic measurements.

In [39]:
# ================================================================
# PHYSICAL CONSTANTS (SI)
# ================================================================
E_CHARGE = 1.602176634e-19    # elementary charge (C)
M_E      = 9.1093837015e-31   # electron mass (kg)
C_LIGHT  = 299792458.0        # speed of light (m/s)
MU_0     = 1.25663706212e-6   # vacuum permeability (H/m)

# ================================================================
# ACCELERATING VOLTAGE → K_v (KNOWN, NOT FITTED)
# ================================================================
U_ACCEL = 200e3  # 200 kV

gamma_rel  = 1 + E_CHARGE * U_ACCEL / (M_E * C_LIGHT**2)
v_electron = C_LIGHT * np.sqrt(1 - 1 / gamma_rel**2)
K_V        = E_CHARGE * MU_0 / (2 * M_E * v_electron)   # rad / AT

print(f"Accelerating voltage: U  = {U_ACCEL/1e3:.0f} kV")
print(f"Relativistic gamma:   γ  = {gamma_rel:.4f}")
print(f"Electron velocity:    v  = {v_electron:.4e} m/s  ({v_electron/C_LIGHT:.4f} c)")
print(f"Rotation constant:    Kv = {K_V:.6e} rad/AT")

# ================================================================
# TRUE SYSTEM PARAMETERS (2-LENS)
# ================================================================
N_LENSES = 2

# Propagation distances (m)
D_TRUE = np.array([2e-3, 40e-3, 200e-3])

# Per-lens excitation (ampere-turns) and geometry constants
I0_TRUE = np.array([2000.0, 800.0])
CF_TRUE = np.array([8.333e-5, 3.125e-5])   # 1 / (m · AT²)

# Derived quantities
F_DERIVED  = 1.0 / (CF_TRUE * I0_TRUE**2)
PSI_DERIVED = K_V * I0_TRUE

# Wobble / defocus grids
WOBBLE_VALUES    = np.array([-0.01, 0.0, 0.01])
Z_DEFOCUS_VALUES = np.array([0.0, 30e-3, 60e-3])

print("\n" + "=" * 60)
print("TRUE SYSTEM PARAMETERS")
print("=" * 60)
print(f"Distances: {D_TRUE*1e3} mm")
for i in range(N_LENSES):
    print(f"  Lens {i+1}:  I0 = {I0_TRUE[i]:.0f} AT,  Cf = {CF_TRUE[i]:.3e}"
          f"  →  f = {F_DERIVED[i]*1e3:.2f} mm,  ψ = {np.degrees(PSI_DERIVED[i]):.1f}°")
print(f"\nWobbles: {WOBBLE_VALUES}")
print(f"Defocus: {Z_DEFOCUS_VALUES*1e3} mm")

Accelerating voltage: U  = 200 kV
Relativistic gamma:   γ  = 1.3914
Electron velocity:    v  = 2.0845e+08 m/s  (0.6953 c)
Rotation constant:    Kv = 5.301506e-04 rad/AT

TRUE SYSTEM PARAMETERS
Distances: [  2.  40. 200.] mm
  Lens 1:  I0 = 2000 AT,  Cf = 8.333e-05  →  f = 3.00 mm,  ψ = 60.8°
  Lens 2:  I0 = 800 AT,  Cf = 3.125e-05  →  f = 50.00 mm,  ψ = 24.3°

Wobbles: [-0.01  0.    0.01]
Defocus: [ 0. 30. 60.] mm


## Forward Model: ABCD + Rotation

Compute $A,B$ from a chain of propagation and thin-lens matrices, and compute rotation from the $\theta(I)$ model.

In [40]:
def f_from_excitation(Cf_i, I0_i, w):
    """Focal length from Glaser Eq. 16: 1/f = Cf * (I0*(1+w))^2."""
    return 1.0 / (Cf_i * (I0_i * (1.0 + w)) ** 2)


def psi_from_excitation(Kv, I0_i, w):
    """Image rotation from Glaser Eq. 17: ψ = Kv * I0 * (1+w)."""
    return Kv * I0_i * (1.0 + w)


def build_abcd(dists, focals, xp=jnp):
    """Construct ABCD matrix for N-lens system.

    dists: length N+1, focals: length N
    M = P(d_{N+1}) L_N ... L_1 P(d_1)
    """
    M = propagation_matrix(dists[-1], xp=xp)
    for i in reversed(range(len(focals))):
        M = M @ lens_matrix(focals[i], xp=xp)
        M = M @ propagation_matrix(dists[i], xp=xp)
    return M


def compute_AB(dists, focals):
    M = build_abcd(dists, focals, xp=jnp)
    return M[0, 0], M[0, 1]


def model_measurement(dists, I0, Cf, Kv, wobble_lens, w, defocus):
    """Forward model: A, B, ψ from Glaser bell parametrisation.

    Pure JAX — compiled via vmap + Optimistix.

    Args:
        dists: propagation distances (N+1,)
        I0: per-lens excitation currents (N,)
        Cf: per-lens geometry constants (N,)
        Kv: rotation constant (scalar, known from voltage)
        wobble_lens: which lens index to wobble
        w: wobble value (scalar)
        defocus: detector defocus (scalar)
    """
    # Focal lengths
    f_use = []
    for i in range(len(I0)):
        w_i = jnp.where(i == wobble_lens, w, 0.0)
        f_use.append(f_from_excitation(Cf[i], I0[i], w_i))
    f_use = jnp.array(f_use)

    # Distances (add defocus to last segment)
    d_use = jnp.array(dists)
    d_use = d_use.at[-1].add(defocus)

    A, B = compute_AB(d_use, f_use)

    # Total rotation (sum over all lenses)
    psi_total = 0.0
    for i in range(len(I0)):
        w_i = jnp.where(i == wobble_lens, w, 0.0)
        psi_total = psi_total + psi_from_excitation(Kv, I0[i], w_i)

    return A, B, psi_total


print("Forward model ready (Glaser bell parametrisation)")

Forward model ready (Glaser bell parametrisation)


## Sympy 5×5 Matrix (Glaser Physics)

Symbolic 5×5 transfer matrix using:

- $1/f_i = C_{f,i} \cdot I_i^2$ (Eq. 16)
- $\psi_i = K_v \cdot I_i$ (Eq. 17)
- Wobble: $I_i = I_{0,i}(1 + w_i)$

In [41]:
# Sympy definitions — Glaser parametrisation
Cf1_s, Cf2_s = sp.symbols("C_{f1} C_{f2}", positive=True)
Kv_s = sp.symbols("K_v", positive=True)
I1_0, I2_0 = sp.symbols("I_{01} I_{02}", positive=True)
w1, w2 = sp.symbols("w_1 w_2")
d1, d2, d3 = sp.symbols("d_1 d_2 d_3", positive=True)

# Currents with wobble
I1 = I1_0 * (1 + w1)
I2 = I2_0 * (1 + w2)

# Focal lengths: 1/f = Cf * I^2  (Eq. 16)
f1 = 1 / (Cf1_s * I1**2)
f2 = 1 / (Cf2_s * I2**2)

# Rotations: ψ = Kv * I  (Eq. 17)
psi1 = Kv_s * I1
psi2 = Kv_s * I2


def propagation_5x5(d):
    return sp.Matrix([
        [1, 0, d, 0, 0],
        [0, 1, 0, d, 0],
        [0, 0, 1, 0, 0],
        [0, 0, 0, 1, 0],
        [0, 0, 0, 0, 1],
    ])


def lens_5x5(f):
    return sp.Matrix([
        [1, 0, 0, 0, 0],
        [0, 1, 0, 0, 0],
        [-1 / f, 0, 1, 0, 0],
        [0, -1 / f, 0, 1, 0],
        [0, 0, 0, 0, 1],
    ])


def rotation_5x5(theta):
    c = sp.cos(theta)
    s = sp.sin(theta)
    return sp.Matrix([
        [c, -s, 0, 0, 0],
        [s,  c, 0, 0, 0],
        [0,  0, c, -s, 0],
        [0,  0, s,  c, 0],
        [0,  0, 0,  0, 1],
    ])


M5 = (
    propagation_5x5(d3)
    * rotation_5x5(psi2)
    * lens_5x5(f2)
    * propagation_5x5(d2)
    * rotation_5x5(psi1)
    * lens_5x5(f1)
    * propagation_5x5(d1)
)

print("Glaser bell model relations:")
display(sp.Eq(sp.Symbol("1/f_1"), 1/f1))
display(sp.Eq(sp.Symbol("1/f_2"), 1/f2))
display(sp.Eq(sp.Symbol("ψ_1"), psi1))
display(sp.Eq(sp.Symbol("ψ_2"), psi2))

print("\nKey: K_v is KNOWN from voltage; C_{f,i} are per-lens geometry constants")
print("\n5×5 transfer matrix:")
display(M5)

Glaser bell model relations:


Eq(1/f_1, C_{f1}*I_{01}**2*(w_1 + 1)**2)

Eq(1/f_2, C_{f2}*I_{02}**2*(w_2 + 1)**2)

Eq(ψ_1, I_{01}*K_v*(w_1 + 1))

Eq(ψ_2, I_{02}*K_v*(w_2 + 1))


Key: K_v is KNOWN from voltage; C_{f,i} are per-lens geometry constants

5×5 transfer matrix:


Matrix([
[  -C_{f1}*I_{01}**2*(w_1 + 1)**2*((d_2*(C_{f2}*I_{02}**2*d_3*(w_2 + 1)**2*sin(I_{02}*K_v*(w_2 + 1)) - sin(I_{02}*K_v*(w_2 + 1))) - d_3*sin(I_{02}*K_v*(w_2 + 1)))*sin(I_{01}*K_v*(w_1 + 1)) + (d_2*(-C_{f2}*I_{02}**2*d_3*(w_2 + 1)**2*cos(I_{02}*K_v*(w_2 + 1)) + cos(I_{02}*K_v*(w_2 + 1))) + d_3*cos(I_{02}*K_v*(w_2 + 1)))*cos(I_{01}*K_v*(w_1 + 1))) + (C_{f2}*I_{02}**2*d_3*(w_2 + 1)**2*sin(I_{02}*K_v*(w_2 + 1)) - sin(I_{02}*K_v*(w_2 + 1)))*sin(I_{01}*K_v*(w_1 + 1)) + (-C_{f2}*I_{02}**2*d_3*(w_2 + 1)**2*cos(I_{02}*K_v*(w_2 + 1)) + cos(I_{02}*K_v*(w_2 + 1)))*cos(I_{01}*K_v*(w_1 + 1)),    -C_{f1}*I_{01}**2*(w_1 + 1)**2*((d_2*(C_{f2}*I_{02}**2*d_3*(w_2 + 1)**2*sin(I_{02}*K_v*(w_2 + 1)) - sin(I_{02}*K_v*(w_2 + 1))) - d_3*sin(I_{02}*K_v*(w_2 + 1)))*cos(I_{01}*K_v*(w_1 + 1)) - (d_2*(-C_{f2}*I_{02}**2*d_3*(w_2 + 1)**2*cos(I_{02}*K_v*(w_2 + 1)) + cos(I_{02}*K_v*(w_2 + 1))) + d_3*cos(I_{02}*K_v*(w_2 + 1)))*sin(I_{01}*K_v*(w_1 + 1))) + (C_{f2}*I_{02}**2*d_3*(w_2 + 1)**2*sin(I_{02}*K_v*(w_2 + 

## Generate Synthetic Measurements

Compute synthetic $A,B,\theta$ for each wobble setting (one lens at a time).

In [42]:
def generate_measurements(d_true, I0_true, Cf_true, Kv, wobble_values, defocus_values):
    """Generate synthetic measurements as vectorized JAX arrays.

    Returns dict: {wobble_lens, wobble, defocus, A, B, psi}
    """
    wl_list, w_list, df_list = [], [], []
    A_list, B_list, psi_list = [], [], []

    for lens_idx in range(len(I0_true)):
        for w in wobble_values:
            for defocus in defocus_values:
                A, B, psi = model_measurement(
                    d_true, I0_true, Cf_true, Kv, lens_idx, w, defocus
                )
                wl_list.append(lens_idx)
                w_list.append(float(w))
                df_list.append(float(defocus))
                A_list.append(float(A))
                B_list.append(float(B))
                psi_list.append(float(psi))

    return {
        "wobble_lens": jnp.array(wl_list, dtype=jnp.int32),
        "wobble": jnp.array(w_list),
        "defocus": jnp.array(df_list),
        "A": jnp.array(A_list),
        "B": jnp.array(B_list),
        "psi": jnp.array(psi_list),
    }


measurements = generate_measurements(
    D_TRUE, I0_TRUE, CF_TRUE, K_V, WOBBLE_VALUES, Z_DEFOCUS_VALUES
)

A_vals   = np.array(measurements["A"])
B_vals   = np.array(measurements["B"])
PSI_vals = np.array(measurements["psi"])

A_SCALE   = max(np.max(np.abs(A_vals)), 1e-12)
B_SCALE   = max(np.max(np.abs(B_vals)), 1e-12)
PSI_SCALE = max(np.max(np.abs(PSI_vals)), 1e-12)

print(f"Generated {len(A_vals)} measurements")
print(f"  A range:   [{A_vals.min():.3e}, {A_vals.max():.3e}]")
print(f"  B range:   [{B_vals.min():.3e}, {B_vals.max():.3e}]")
print(f"  ψ range:   [{PSI_vals.min():.4f}, {PSI_vals.max():.4f}] rad"
      f"  ({np.degrees(PSI_vals.min()):.1f}° – {np.degrees(PSI_vals.max()):.1f}°)")

Generated 18 measurements
  A range:   [-3.614e+01, -2.867e+01]
  B range:   [1.944e-02, 2.386e-02]
  ψ range:   [1.4738, 1.4950] rad  (84.4° – 85.7°)


## Residuals for Least Squares

Define residuals for $A$, $B$, and $\theta$ and build a loss function.

In [43]:
def make_residual_fn(measurements, n_lenses, scales, Kv):
    """vmap-vectorized residual function (Glaser parametrisation).

    Parameter vector: [d1,...,d_{N+1}, I0_1,...,I0_N, Cf_1,...,Cf_N]
    Total: 3N+1 parameters.

    K_v is KNOWN (not fitted) — eliminates the f ↔ ψ degeneracy.
    """
    meas_wl  = measurements["wobble_lens"]
    meas_w   = measurements["wobble"]
    meas_df  = measurements["defocus"]
    meas_A   = measurements["A"]
    meas_B   = measurements["B"]
    meas_psi = measurements["psi"]

    A_scale, B_scale, psi_scale = scales
    n_dist = n_lenses + 1
    n_I0   = n_lenses

    @jax.jit
    def residual_fn(params):
        d  = params[:n_dist]
        I0 = params[n_dist : n_dist + n_I0]
        Cf = params[n_dist + n_I0 : n_dist + 2 * n_I0]

        def single_pred(wl, w, df):
            return model_measurement(d, I0, Cf, Kv, wl, w, df)

        A_pred, B_pred, psi_pred = jax.vmap(single_pred)(meas_wl, meas_w, meas_df)

        res_A   = (A_pred   - meas_A)   / A_scale
        res_B   = (B_pred   - meas_B)   / B_scale
        res_psi = (psi_pred - meas_psi) / psi_scale

        return jnp.concatenate([res_A, res_B, res_psi])

    return residual_fn


def make_loss_fn(measurements, n_lenses, scales, Kv):
    """Loss = sum of squared residuals."""
    residual_fn = make_residual_fn(measurements, n_lenses, scales, Kv)

    def loss_fn(params):
        r = residual_fn(params)
        return jnp.sum(r ** 2)

    return loss_fn


scales = (A_SCALE, B_SCALE, PSI_SCALE)
residual_fn = make_residual_fn(measurements, N_LENSES, scales, K_V)
loss_fn     = make_loss_fn(measurements, N_LENSES, scales, K_V)

x_true    = np.concatenate([D_TRUE, I0_TRUE, CF_TRUE])
loss_true = loss_fn(x_true)

print(f"Parameter vector: [d (×{N_LENSES+1}), I0 (×{N_LENSES}), Cf (×{N_LENSES})]")
print(f"Total: {len(x_true)} parameters, {len(A_vals)} measurements")
print(f"Loss at true parameters: {loss_true:.6e}")

Parameter vector: [d (×3), I0 (×2), Cf (×2)]
Total: 7 parameters, 18 measurements
Loss at true parameters: 0.000000e+00


In [44]:

def make_residual_fn_constrained(measurements, n_lenses, scales, Kv, total_distance=None):
    """vmap-vectorized residual function with optional total distance constraint.

    If total_distance is specified:
        Parameter vector: [d1,...,d_N, I0_1,...,I0_N, Cf_1,...,Cf_N]
        where d_{N+1} = total_distance - sum(d_1...d_N)
        Total: 3N parameters (one less distance)
    
    If total_distance is None:
        Parameter vector: [d1,...,d_{N+1}, I0_1,...,I0_N, Cf_1,...,Cf_N]
        Total: 3N+1 parameters (standard)

    K_v is KNOWN (not fitted) — eliminates the f ↔ ψ degeneracy.
    """
    meas_wl  = measurements["wobble_lens"]
    meas_w   = measurements["wobble"]
    meas_df  = measurements["defocus"]
    meas_A   = measurements["A"]
    meas_B   = measurements["B"]
    meas_psi = measurements["psi"]

    A_scale, B_scale, psi_scale = scales
    n_I0   = n_lenses
    
    if total_distance is not None:
        n_dist_fit = n_lenses  # Only fit N distances, derive the (N+1)th
    else:
        n_dist_fit = n_lenses + 1  # Fit all N+1 distances

    @jax.jit
    def residual_fn(params):
        if total_distance is not None:
            # Reconstruct full distance array from first N + constraint
            d_fit = params[:n_dist_fit]
            d_last = total_distance - jnp.sum(d_fit)
            d = jnp.concatenate([d_fit, jnp.array([d_last])])
        else:
            d = params[:n_dist_fit]
        
        I0 = params[n_dist_fit : n_dist_fit + n_I0]
        Cf = params[n_dist_fit + n_I0 : n_dist_fit + 2 * n_I0]

        def single_pred(wl, w, df):
            return model_measurement(d, I0, Cf, Kv, wl, w, df)

        A_pred, B_pred, psi_pred = jax.vmap(single_pred)(meas_wl, meas_w, meas_df)

        res_A   = (A_pred   - meas_A)   / A_scale
        res_B   = (B_pred   - meas_B)   / B_scale
        res_psi = (psi_pred - meas_psi) / psi_scale

        return jnp.concatenate([res_A, res_B, res_psi])

    return residual_fn


## Run Optimization with Levenberg-Marquardt

Use scipy's least_squares with Levenberg-Marquardt method to fit per-lens rotation and nonlinearity constants.


In [45]:
print("=" * 70)
print("FITTING 2-LENS SYSTEM (Glaser Parametrisation)")
print("=" * 70)

x_true = np.concatenate([D_TRUE, I0_TRUE, CF_TRUE])

# Generate random initial guesses with WIDE ranges
rng = np.random.default_rng(90)
n_dist = N_LENSES + 1
n_I0 = N_LENSES
n_Cf = N_LENSES

# Distances: [0.5 mm, 300 mm]
x0_dist = 0.5e-3 + (300e-3 - 0.5e-3) * rng.random(n_dist)
# Excitations: [100 AT, 10000 AT]
x0_I0 = 100.0 + (10000.0 - 100.0) * rng.random(n_I0)
# Geometry constants: log-uniform [1e-6, 1e-4]
x0_Cf = 10.0 ** (np.log10(1e-6) + (np.log10(1e-4) - np.log10(1e-6)) * rng.random(n_Cf))

x0 = np.concatenate([x0_dist, x0_I0, x0_Cf])

print(f"True parameters:")
print(f"  d:  {D_TRUE*1e3} mm")
print(f"  I0: {I0_TRUE} AT")
print(f"  Cf: {CF_TRUE}")

print(f"\nRandom initial guess (wide ranges):")
print(f"  d:  {x0_dist*1e3} mm")
print(f"  I0: {x0_I0} AT")
print(f"  Cf: {x0_Cf}")

# Define bounds
lower_bounds = np.concatenate([
    np.full(n_dist, 1e-4),  # min distance 0.1 mm
    np.full(n_I0, 10.0),    # min current 10 AT
    np.full(n_Cf, 1e-7),    # min Cf
])
upper_bounds = np.concatenate([
    np.full(n_dist, 1.0),      # max distance 1000 mm
    np.full(n_I0, 100000.0),   # max current 100k AT
    np.full(n_Cf, 1e-3),       # max Cf
])

print(f"\nOptimizing with scipy.optimize.least_squares (dogbox with bounds)...")
start_time = time.time()

sol_scipy = scipy_least_squares(
    fun=residual_fn,
    x0=x0,
    bounds=(lower_bounds, upper_bounds),
    method='dogbox',  # Dogleg with box constraints - faster than TRF
    ftol=1e-10,
    xtol=1e-10,
    gtol=1e-10,
    max_nfev=15000,
    verbose=0,
)

elapsed = time.time() - start_time

# Results
x_fitted = sol_scipy.x
n_dist = N_LENSES + 1

d_fit  = x_fitted[:n_dist]
I0_fit = x_fitted[n_dist : n_dist + N_LENSES]
Cf_fit = x_fitted[n_dist + N_LENSES : n_dist + 2 * N_LENSES]

# Derived focal lengths
f_true = 1.0 / (CF_TRUE * I0_TRUE**2)
f_fit  = 1.0 / (Cf_fit * I0_fit**2)

errors = np.abs((x_fitted - x_true) / (x_true + 1e-30)) * 100
r_final = np.array(residual_fn(x_fitted))

print(f"\n{'='*70}")
print(f"RESULTS")
print(f"{'='*70}")
print(f"Converged: {sol_scipy.success}")
print(f"Message: {sol_scipy.message}")
print(f"Loss: {np.sum(r_final**2):.6e}")
print(f"Time: {elapsed:.2f}s")

param_names = (
    [f"d{i+1}" for i in range(n_dist)]
    + [f"I0_{i+1}" for i in range(N_LENSES)]
    + [f"Cf_{i+1}" for i in range(N_LENSES)]
)

print(f"\n{'Param':>8s} {'True':>12s} {'Fitted':>12s} {'Error':>8s}")
print("-" * 45)
for name, tv, fv, err in zip(param_names, x_true, x_fitted, errors):
    s = "✓" if err < 1.0 else ("△" if err < 5.0 else "✗")
    print(f"{name:>8s} {tv:>12.4e} {fv:>12.4e} {err:>7.2f}% {s}")

print(f"\nDerived focal lengths:")
for i in range(N_LENSES):
    print(f"  f{i+1}: {f_true[i]*1e3:.3f} mm (true) → {f_fit[i]*1e3:.3f} mm (fitted)")

print(f"\nMax error: {np.max(errors):.2f}%")

FITTING 2-LENS SYSTEM (Glaser Parametrisation)
True parameters:
  d:  [  2.  40. 200.] mm
  I0: [2000.  800.] AT
  Cf: [8.333e-05 3.125e-05]

Random initial guess (wide ranges):
  d:  [112.01366756 204.72818371 155.79875206] mm
  I0: [4241.66713684 6993.48852965] AT
  Cf: [1.78148034e-06 2.51009193e-05]

Optimizing with scipy.optimize.least_squares (dogbox with bounds)...

RESULTS
Converged: True
Message: `xtol` termination condition is satisfied.
Loss: 1.352474e+04
Time: 0.25s

   Param         True       Fitted    Error
---------------------------------------------
      d1   2.0000e-03   8.5837e-02 4191.86% ✗
      d2   4.0000e-02   1.0000e-04   99.75% ✗
      d3   2.0000e-01   1.5545e-01   22.27% ✗
    I0_1   2.0000e+03   1.9999e+03    0.00% ✓
    I0_2   8.0000e+02   8.0010e+02    0.01% ✓
    Cf_1   8.3330e-05   1.0000e-07   99.88% ✗
    Cf_2   3.1250e-05   8.7269e-05  179.26% ✗

Derived focal lengths:
  f1: 3.000 mm (true) → 2500.246 mm (fitted)
  f2: 50.000 mm (true) → 17.900 mm 

## 3-Lens System Test

Test the Glaser parametrisation on a 3-lens system where each lens has a **different** geometry constant $C_{f,i}$.

In [46]:
# 3-lens system — each lens has different geometry
D_TRUE_3  = np.array([2e-3, 20e-3, 40e-3, 200e-3])
I0_TRUE_3 = np.array([2000.0, 1000.0, 600.0])      # AT
CF_TRUE_3 = np.array([8.333e-5, 5.0e-5, 3.472e-5])  # all different

F_DERIVED_3 = 1.0 / (CF_TRUE_3 * I0_TRUE_3**2)

print("=" * 70)
print("3-LENS SYSTEM (Glaser Parametrisation)")
print("=" * 70)
print(f"Distances: {D_TRUE_3*1e3} mm")
for i in range(3):
    psi_i = K_V * I0_TRUE_3[i]
    print(f"  Lens {i+1}:  I0={I0_TRUE_3[i]:.0f} AT,  Cf={CF_TRUE_3[i]:.3e}"
          f"  →  f={F_DERIVED_3[i]*1e3:.2f} mm,  ψ={np.degrees(psi_i):.1f}°")

x_true_3 = np.concatenate([D_TRUE_3, I0_TRUE_3, CF_TRUE_3])

# Generate SMARTER initial guesses: perturb true values by wider range
rng = np.random.default_rng(43)
n_dist_3 = 4
n_I0_3 = 3
n_Cf_3 = 3

# Perturb true values by factor of 0.05 to 3.0 (challenging range, ±150%)
x0_dist_3 = D_TRUE_3 * (0.05 + rng.random(n_dist_3) * 5)
x0_I0_3 = I0_TRUE_3 * (0.05 + rng.random(n_I0_3) * 5)
x0_Cf_3 = CF_TRUE_3 * (0.05 + rng.random(n_Cf_3) * 5)

x0_3 = np.concatenate([x0_dist_3, x0_I0_3, x0_Cf_3])

print(f"Initial guess (perturbed true values ±150%, range [0.05 to 3.0]×true):")
print(f"  d:  {x0_dist_3*1e3} mm")
print(f"  I0: {x0_I0_3} AT")
print(f"  Cf: {x0_Cf_3}")

print(f"\nParameters: {len(x_true_3)}  (4 dist + 3 I0 + 3 Cf)")

# ================================================================
# CONSTRAINT OPTION: Known Total Distance
# ================================================================
TOTAL_DISTANCE_KNOWN = np.sum(D_TRUE_3)  # Set to None to disable constraint
# Example: TOTAL_DISTANCE_KNOWN = 0.262  # 262 mm total column distance
if TOTAL_DISTANCE_KNOWN is not None:
    n_params_with_constraint = 3 * 3 + 3  # 3 (I0) + 3 (Cf) + 3 (distances, last one derived)
    print(f"\n>>> Using CONSTRAINT: Known total distance = {TOTAL_DISTANCE_KNOWN*1000:.1f} mm")
    print(f"    Parameters reduced to {n_params_with_constraint}  (3 dist + 3 I0 + 3 Cf)")
else:
    print(f"\n>>> NO CONSTRAINT: Fitting all {len(x_true_3)} parameters")

wobble_grids = {
    "sparse": np.array([-0.02, 0.0, 0.02]),
    #"full":   np.array([-0.05, -0.03, -0.01, 0.0, 0.01, 0.03, 0.05]),
}
defocus_grids = {
    #"minimal": np.array([0.0]),
    "full":    np.array([0.0, 30e-3, 60e-3]),
}

print(f"\n{'Wobble':>8s} {'Defocus':>8s} {'Meas':>5s} {'Max Err':>10s} {'Loss':>12s} {'Time':>7s}")
print("-" * 60)

for wn, ws in wobble_grids.items():
    for dn, ds in defocus_grids.items():
        m = generate_measurements(D_TRUE_3, I0_TRUE_3, CF_TRUE_3, K_V, ws, ds)
        sc = (
            max(np.max(np.abs(m["A"])),   1e-12),
            max(np.max(np.abs(m["B"])),   1e-12),
            max(np.max(np.abs(m["psi"])), 1e-12),
        )
        
        # Choose between constrained and unconstrained residual function
        if TOTAL_DISTANCE_KNOWN is not None:
            rfn = make_residual_fn_constrained(m, 3, sc, K_V, total_distance=TOTAL_DISTANCE_KNOWN)
            # For constrained: x0 has only 3 distances + 3 I0 + 3 Cf (9 params total)
            x0_use = np.concatenate([x0_dist_3[:3], x0_I0_3, x0_Cf_3])
            n_dist_use = 3
            n_bounds = n_I0_3 + n_Cf_3 + 3  # 3 distances fit, rest as before
        else:
            rfn = make_residual_fn(m, 3, sc, K_V)
            # For unconstrained: x0 has 4 distances + 3 I0 + 3 Cf (10 params total)
            x0_use = x0_3
            n_dist_use = 4
            n_bounds = n_I0_3 + n_Cf_3 + 4
        
        # Define bounds for 3-lens
        lower_bounds_3 = np.concatenate([
            np.full(n_dist_use, 1e-4),  # min distance 0.1 mm
            np.full(n_I0_3, 10.0),    # min current 10 AT
            np.full(n_Cf_3, 1e-7),    # min Cf
        ])
        upper_bounds_3 = np.concatenate([
            np.full(n_dist_use, 1.0),      # max distance 1000 mm
            np.full(n_I0_3, 100000.0),   # max current 100k AT
            np.full(n_Cf_3, 1e-3),       # max Cf
        ])
        
        # Optimize with dogbox (fast)
        t0 = time.time()
        sol_scipy_3 = scipy_least_squares(
            fun=rfn,
            x0=x0_use,
            bounds=(lower_bounds_3, upper_bounds_3),
            method='dogbox',  # Dogleg with box constraints - faster
            ftol=1e-11,
            xtol=1e-11,
            gtol=1e-11,
            max_nfev=30000,
            verbose=0,
        )
        elapsed_3 = time.time() - t0
        
        x_fit = np.array(sol_scipy_3.x)
        r = np.array(rfn(sol_scipy_3.x))
        
        # For error calculation, reconstruct full parameter vector if constrained
        if TOTAL_DISTANCE_KNOWN is not None:
            x_fit_full = np.concatenate([x_fit[:3], np.array([TOTAL_DISTANCE_KNOWN - np.sum(x_fit[:3])]), x_fit[3:]])
            errs = np.abs((x_fit_full - x_true_3) / (x_true_3 + 1e-30)) * 100
        else:
            errs = np.abs((x_fit - x_true_3) / (x_true_3 + 1e-30)) * 100
        
        loss = np.sum(r**2)
        n_meas = len(ws) * len(ds) * 3
        s = "✓" if np.max(errs) < 1.0 else ("△" if np.max(errs) < 50. else "✗")
        print(f"{wn:>8s} {dn:>8s} {n_meas:>5d}   {np.max(errs):>7.1f}% {s} {loss:>12.3e} {elapsed_3:>6.2f}s")

3-LENS SYSTEM (Glaser Parametrisation)
Distances: [  2.  20.  40. 200.] mm
  Lens 1:  I0=2000 AT,  Cf=8.333e-05  →  f=3.00 mm,  ψ=60.8°
  Lens 2:  I0=1000 AT,  Cf=5.000e-05  →  f=20.00 mm,  ψ=30.4°
  Lens 3:  I0=600 AT,  Cf=3.472e-05  →  f=80.01 mm,  ψ=18.2°
Initial guess (perturbed true values ±150%, range [0.05 to 3.0]×true):
  d:  [  6.62299263   5.37753236   6.00591737 849.21258251] mm
  I0: [5971.43047588 1173.52615102 2285.37681546] AT
  Cf: [1.14033854e-04 1.07494477e-04 8.00350488e-05]

Parameters: 10  (4 dist + 3 I0 + 3 Cf)

>>> Using CONSTRAINT: Known total distance = 262.0 mm
    Parameters reduced to 12  (3 dist + 3 I0 + 3 Cf)

  Wobble  Defocus  Meas    Max Err         Loss    Time
------------------------------------------------------------
  sparse     full    27       0.0% ✓    1.006e-28   0.53s


## 4-Lens System Test

Test the Glaser parametrisation on a 4-lens system with challenging initialization.


In [47]:
# 4-lens system
D_TRUE_4  = np.array([2e-3, 10e-3, 20e-3, 40e-3, 200e-3])
I0_TRUE_4 = np.array([2000.0, 1200.0, 800.0, 500.0])      # AT
CF_TRUE_4 = np.array([8.333e-5, 5.787e-5, 3.906e-5, 2.441e-5])  # all different

F_DERIVED_4 = 1.0 / (CF_TRUE_4 * I0_TRUE_4**2)

print("=" * 70)
print("4-LENS SYSTEM (Glaser Parametrisation)")
print("=" * 70)
print(f"Distances: {D_TRUE_4*1e3} mm")
for i in range(4):
    psi_i = K_V * I0_TRUE_4[i]
    print(f"  Lens {i+1}:  I0={I0_TRUE_4[i]:.0f} AT,  Cf={CF_TRUE_4[i]:.3e}"
          f"  →  f={F_DERIVED_4[i]*1e3:.2f} mm,  ψ={np.degrees(psi_i):.1f}°")

x_true_4 = np.concatenate([D_TRUE_4, I0_TRUE_4, CF_TRUE_4])

# Generate challenging initial guesses
rng = np.random.default_rng(44)
n_dist_4 = 5
n_I0_4 = 4
n_Cf_4 = 4

# Tighter range for higher-dimensional system (4 lenses: [0.25 to 2.25], ±100%)
x0_dist_4 = D_TRUE_4 * (0.25 + rng.random(n_dist_4) * 2.0)
x0_I0_4 = I0_TRUE_4 * (0.25 + rng.random(n_I0_4) * 2.0)
x0_Cf_4 = CF_TRUE_4 * (0.25 + rng.random(n_Cf_4) * 2.0)

x0_4 = np.concatenate([x0_dist_4, x0_I0_4, x0_Cf_4])

print(f"\nInitial guess (perturbed true values ±100%, range [0.25 to 2.25]×true):")
print(f"  d:  {x0_dist_4*1e3} mm")
print(f"  I0: {x0_I0_4} AT")
print(f"  Cf: {x0_Cf_4}")

print(f"\nParameters: {len(x_true_4)}  (5 dist + 4 I0 + 4 Cf)")

# CONSTRAINT: Known total distance
TOTAL_DISTANCE_KNOWN_4 = np.sum(D_TRUE_4)
if TOTAL_DISTANCE_KNOWN_4 is not None:
    print(f"\n>>> Using CONSTRAINT: Known total distance = {TOTAL_DISTANCE_KNOWN_4*1000:.1f} mm")
    print(f"    Parameters reduced to 12  (4 dist + 4 I0 + 4 Cf)")
else:
    print(f"\n>>> NO CONSTRAINT: Fitting all {len(x_true_4)} parameters")

wobble_grids_4 = {
    "sparse": np.array([-0.02, 0.0, 0.02]),
    "full":   np.array([-0.05, -0.03, -0.01, 0.0, 0.01, 0.03, 0.05]),
}
defocus_grids_4 = {
    "minimal": np.array([0.0]),
    "full":    np.array([0.0, 30e-3, 60e-3]),
}

print(f"\n{'Wobble':>8s} {'Defocus':>8s} {'Meas':>5s} {'Max Err':>10s} {'Loss':>12s} {'Time':>7s}")
print("-" * 60)

for wn, ws in wobble_grids_4.items():
    for dn, ds in defocus_grids_4.items():
        m = generate_measurements(D_TRUE_4, I0_TRUE_4, CF_TRUE_4, K_V, ws, ds)
        sc = (
            max(np.max(np.abs(m["A"])),   1e-12),
            max(np.max(np.abs(m["B"])),   1e-12),
            max(np.max(np.abs(m["psi"])), 1e-12),
        )
        
        if TOTAL_DISTANCE_KNOWN_4 is not None:
            rfn = make_residual_fn_constrained(m, 4, sc, K_V, total_distance=TOTAL_DISTANCE_KNOWN_4)
            x0_use = np.concatenate([x0_dist_4[:4], x0_I0_4, x0_Cf_4])
            n_dist_use = 4
        else:
            rfn = make_residual_fn(m, 4, sc, K_V)
            x0_use = x0_4
            n_dist_use = 5
        
        lower_bounds_4 = np.concatenate([
            np.full(n_dist_use, 1e-4),
            np.full(n_I0_4, 10.0),
            np.full(n_Cf_4, 1e-7),
        ])
        upper_bounds_4 = np.concatenate([
            np.full(n_dist_use, 1.0),
            np.full(n_I0_4, 100000.0),
            np.full(n_Cf_4, 1e-3),
        ])
        
        t0 = time.time()
        sol_scipy_4 = scipy_least_squares(
            fun=rfn,
            x0=x0_use,
            bounds=(lower_bounds_4, upper_bounds_4),
            method='dogbox',
            ftol=1e-11,
            xtol=1e-11,
            gtol=1e-11,
            max_nfev=30000,
            verbose=0,
        )
        elapsed_4 = time.time() - t0
        
        x_fit = np.array(sol_scipy_4.x)
        r = np.array(rfn(sol_scipy_4.x))
        
        if TOTAL_DISTANCE_KNOWN_4 is not None:
            x_fit_full = np.concatenate([x_fit[:4], np.array([TOTAL_DISTANCE_KNOWN_4 - np.sum(x_fit[:4])]), x_fit[4:]])
            errs = np.abs((x_fit_full - x_true_4) / (x_true_4 + 1e-30)) * 100
        else:
            errs = np.abs((x_fit - x_true_4) / (x_true_4 + 1e-30)) * 100
        
        loss = np.sum(r**2)
        n_meas = len(ws) * len(ds) * 4
        s = "✓" if np.max(errs) < 1.0 else ("△" if np.max(errs) < 50. else "✗")
        print(f"{wn:>8s} {dn:>8s} {n_meas:>5d}   {np.max(errs):>7.1f}% {s} {loss:>12.3e} {elapsed_4:>6.2f}s")


4-LENS SYSTEM (Glaser Parametrisation)
Distances: [  2.  10.  20.  40. 200.] mm
  Lens 1:  I0=2000 AT,  Cf=8.333e-05  →  f=3.00 mm,  ψ=60.8°
  Lens 2:  I0=1200 AT,  Cf=5.787e-05  →  f=12.00 mm,  ψ=36.5°
  Lens 3:  I0=800 AT,  Cf=3.906e-05  →  f=40.00 mm,  ψ=24.3°
  Lens 4:  I0=500 AT,  Cf=2.441e-05  →  f=163.87 mm,  ψ=15.2°

Initial guess (perturbed true values ±100%, range [0.25 to 2.25]×true):
  d:  [  0.99026204   7.6622615   21.23082911  87.53471585 114.92684991] mm
  I0: [3929.17469423  691.30865057  740.73911835  802.72281738] AT
  Cf: [1.23584085e-04 1.24991812e-04 4.19036290e-05 5.19758133e-05]

Parameters: 13  (5 dist + 4 I0 + 4 Cf)

>>> Using CONSTRAINT: Known total distance = 272.0 mm
    Parameters reduced to 12  (4 dist + 4 I0 + 4 Cf)

  Wobble  Defocus  Meas    Max Err         Loss    Time
------------------------------------------------------------
  sparse  minimal    12      99.8% ✗    2.638e+01   0.42s
  sparse     full    36       0.0% ✓    8.140e-28   0.61s
    full

## 5-Lens System Test

Test the Glaser parametrisation on a 5-lens system with challenging initialization.


In [51]:
# 5-lens system
D_TRUE_5  = np.array([2e-3, 8e-3, 15e-3, 25e-3, 40e-3, 200e-3])
I0_TRUE_5 = np.array([2000.0, 1300.0, 900.0, 700.0, 500.0])     # AT
CF_TRUE_5 = np.array([8.333e-5, 5.415e-5, 3.951e-5, 2.755e-5, 2.0e-5])  # all different

F_DERIVED_5 = 1.0 / (CF_TRUE_5 * I0_TRUE_5**2)

print("=" * 70)
print("5-LENS SYSTEM (Glaser Parametrisation)")
print("=" * 70)
print(f"Distances: {D_TRUE_5*1e3} mm")
for i in range(5):
    psi_i = K_V * I0_TRUE_5[i]
    print(f"  Lens {i+1}:  I0={I0_TRUE_5[i]:.0f} AT,  Cf={CF_TRUE_5[i]:.3e}"
          f"  →  f={F_DERIVED_5[i]*1e3:.2f} mm,  ψ={np.degrees(psi_i):.1f}°")

x_true_5 = np.concatenate([D_TRUE_5, I0_TRUE_5, CF_TRUE_5])

# Generate challenging initial guesses
rng = np.random.default_rng(45)
n_dist_5 = 6
n_I0_5 = 5
n_Cf_5 = 5

x0_dist_5 = D_TRUE_5 * (0.25 + rng.random(n_dist_5) * 1.6)
x0_I0_5 = I0_TRUE_5 * (0.25 + rng.random(n_I0_5) * 1.6)
x0_Cf_5 = CF_TRUE_5 * (0.25 + rng.random(n_Cf_5) * 1.6)

x0_5 = np.concatenate([x0_dist_5, x0_I0_5, x0_Cf_5])

print(f"\nInitial guess (perturbed true values ±100%, range [0.25 to 2.25]×true):")
print(f"  d:  {x0_dist_5*1e3} mm")
print(f"  I0: {x0_I0_5} AT")
print(f"  Cf: {x0_Cf_5}")

print(f"\nParameters: {len(x_true_5)}  (6 dist + 5 I0 + 5 Cf)")

# CONSTRAINT: Known total distance
TOTAL_DISTANCE_KNOWN_5 = np.sum(D_TRUE_5)
if TOTAL_DISTANCE_KNOWN_5 is not None:
    print(f"\n>>> Using CONSTRAINT: Known total distance = {TOTAL_DISTANCE_KNOWN_5*1000:.1f} mm")
    print(f"    Parameters reduced to 15  (5 dist + 5 I0 + 5 Cf)")
else:
    print(f"\n>>> NO CONSTRAINT: Fitting all {len(x_true_5)} parameters")

wobble_grids_5 = {
    "full":   np.array([-0.05, -0.03, -0.01, 0.0, 0.01, 0.03, 0.05]),
}
defocus_grids_5 = {
    "full":    np.array([0.0, 30e-3, 60e-3]),
}

print(f"\n{'Wobble':>8s} {'Defocus':>8s} {'Meas':>5s} {'Max Err':>10s} {'Loss':>12s} {'Time':>7s}")
print("-" * 60)

for wn, ws in wobble_grids_5.items():
    for dn, ds in defocus_grids_5.items():
        m = generate_measurements(D_TRUE_5, I0_TRUE_5, CF_TRUE_5, K_V, ws, ds)
        sc = (
            max(np.max(np.abs(m["A"])),   1e-12),
            max(np.max(np.abs(m["B"])),   1e-12),
            max(np.max(np.abs(m["psi"])), 1e-12),
        )
        
        if TOTAL_DISTANCE_KNOWN_5 is not None:
            rfn = make_residual_fn_constrained(m, 5, sc, K_V, total_distance=TOTAL_DISTANCE_KNOWN_5)
            x0_use = np.concatenate([x0_dist_5[:5], x0_I0_5, x0_Cf_5])
            n_dist_use = 5
        else:
            rfn = make_residual_fn(m, 5, sc, K_V)
            x0_use = x0_5
            n_dist_use = 6
        
        lower_bounds_5 = np.concatenate([
            np.full(n_dist_use, 1e-4),
            np.full(n_I0_5, 10.0),
            np.full(n_Cf_5, 1e-7),
        ])
        upper_bounds_5 = np.concatenate([
            np.full(n_dist_use, 1.0),
            np.full(n_I0_5, 100000.0),
            np.full(n_Cf_5, 1e-3),
        ])
        
        t0 = time.time()
        sol_scipy_5 = scipy_least_squares(
            fun=rfn,
            x0=x0_use,
            bounds=(lower_bounds_5, upper_bounds_5),
            method='dogbox',
            ftol=1e-11,
            xtol=1e-11,
            gtol=1e-11,
            max_nfev=30000,
            verbose=0,
        )
        elapsed_5 = time.time() - t0
        
        x_fit = np.array(sol_scipy_5.x)
        r = np.array(rfn(sol_scipy_5.x))
        
        if TOTAL_DISTANCE_KNOWN_5 is not None:
            x_fit_full = np.concatenate([x_fit[:5], np.array([TOTAL_DISTANCE_KNOWN_5 - np.sum(x_fit[:5])]), x_fit[5:]])
            errs = np.abs((x_fit_full - x_true_5) / (x_true_5 + 1e-30)) * 100
        else:
            errs = np.abs((x_fit - x_true_5) / (x_true_5 + 1e-30)) * 100
        
        loss = np.sum(r**2)
        n_meas = len(ws) * len(ds) * 5
        s = "✓" if np.max(errs) < 1.0 else ("△" if np.max(errs) < 50. else "✗")
        print(f"{wn:>8s} {dn:>8s} {n_meas:>5d}   {np.max(errs):>7.1f}% {s} {loss:>12.3e} {elapsed_5:>6.2f}s")


5-LENS SYSTEM (Glaser Parametrisation)
Distances: [  2.   8.  15.  25.  40. 200.] mm
  Lens 1:  I0=2000 AT,  Cf=8.333e-05  →  f=3.00 mm,  ψ=60.8°
  Lens 2:  I0=1300 AT,  Cf=5.415e-05  →  f=10.93 mm,  ψ=39.5°
  Lens 3:  I0=900 AT,  Cf=3.951e-05  →  f=31.25 mm,  ψ=27.3°
  Lens 4:  I0=700 AT,  Cf=2.755e-05  →  f=74.08 mm,  ψ=21.3°
  Lens 5:  I0=500 AT,  Cf=2.000e-05  →  f=200.00 mm,  ψ=15.2°

Initial guess (perturbed true values ±100%, range [0.25 to 2.25]×true):
  d:  [  2.3340181    8.76468666  22.07760576  38.71771068  42.65464673
 299.38869517] mm
  I0: [3047.56257427 1562.09040903  813.35238753  926.82961258  626.97426388] AT
  Cf: [1.32839000e-04 7.62790246e-05 4.33199277e-05 4.93835258e-05
 2.00001346e-05]

Parameters: 16  (6 dist + 5 I0 + 5 Cf)

>>> Using CONSTRAINT: Known total distance = 290.0 mm
    Parameters reduced to 15  (5 dist + 5 I0 + 5 Cf)

  Wobble  Defocus  Meas    Max Err         Loss    Time
------------------------------------------------------------
    full     f

## Summary: Glaser Bell Model Parametrisation

### Physics

Each lens is characterised by two fitted parameters:
- **$I_{0,i}$** — nominal excitation (ampere-turns)
- **$C_{f,i}$** — geometry constant (bore, gap, pole-piece shape), unique to each lens

The accelerating voltage provides the **known** constant $K_v = e\mu_0/(2m_e v)$:
- **Rotation** $\psi_i = K_v \cdot I_{0,i}(1+w)$ → directly constrains $I_{0,i}$
- **Focal length** $1/f_i = C_{f,i} \cdot I_{0,i}^2(1+w)^2$ → uniquely determines $C_{f,i}$

### No degeneracy

Unlike a $(f_0, k_\phi)$ parametrisation, knowing $K_v$ means the rotation measurement independently constrains the excitation current, breaking the $f \leftrightarrow \psi$ degeneracy.

### Parameter count

| System | Distances | $I_0$ | $C_f$ | Total |
|--------|-----------|-------|--------|-------|
| 2-lens | 3 | 2 | 2 | **7** |
| 3-lens | 4 | 3 | 3 | **10** |
| N-lens | N+1 | N | N | **3N+1** |